---
<div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
    <img src='../misc/ufpr.png' style='width: 15%;'>
    <div style="text-align: center; flex-grow: 1;">
        <h2 style='line-height: 0.5; color:#ffda77; margin-top: 50px;'>Universidade Federal do Paraná</h2>
        <h2 style='line-height: 0.5; color:#ffda77;'>LABAP</h2>
        <h6 style='line-height: 0.5; color:#ffda77;'>Laboratório de Análise de Bacias e Petrofísica</h6>
    </div>
    <img src='../misc/labap.png' style='width: 15%;'>
</div>
<br>

---


# Imports

In [ ]:
from pathlib import Path

import lasio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import iqr

from sonic_ml_utils import plot_well

import warnings
warnings.filterwarnings("ignore")

# Loading LAS Data

The well files are in .LAS format, which contains relevant information about the location and drilling of the well, along with the log curves required for this study.

Each file is loaded as a DataFrame and stored in a dictionary keyed by well name.

In [ ]:
LAS_FOLDER = Path('../data/raw/LAS')

# Mnemonic → standard name mapping
# Rule: corrected version takes priority over raw version.
# Rename is applied after loading — IP already exported the correct curves,
# but names in the LAS may vary by tool/vintage.
COLUMN_RENAME = {
    # Gamma Ray
    'ECGR'     : 'GR',   # Spectral GR corrected (Schlumberger) — preferred
    'HSGR'     : 'GR',   # Total spectral GR (Halliburton)
    'ECGR_EDTC': 'GR',   # Corrected GR from EDTC tool
    'RGR'      : 'GR',   # Renamed GR (some LWD wells)
    # Density
    'RHOZ'     : 'RHOB', # Corrected density (Schlumberger)
    'RHOC'     : 'RHOB', # Corrected density (alternative)
    'ZDEN'     : 'RHOB', # Density (Baker Hughes)
    # Neutron porosity
    'TNPH'     : 'NPHI', # Thermal neutron compensated — preferred
    'CNCF'     : 'NPHI', # Corrected compensated neutron
    'CNC'      : 'NPHI', # Raw compensated neutron
    'SPHI'     : 'NPHI', # Sonic porosity (fallback)
    # Deep resistivity
    'AT90'     : 'RT90', # Array induction 90" (Schlumberger)
    'AHT90'    : 'RT90', # Array induction 90" (Halliburton)
    'M2RX'     : 'RT90', # Deep measurement (Baker Hughes)
    'AF90'     : 'RT90', # Array formation 90"
    'A16L'     : 'RT90', # Attenuation resistivity 16" LWD (3-BRSA-1367)
    # Compressional sonic
    'DTCO'     : 'DT',   # Compressional delta-T (Schlumberger)
    'DTC'      : 'DT',   # Compressional delta-T (alternative)
    'DT24'     : 'DT',   # Slowness 24-inch (Baker Hughes)
    # Caliper
    'HCAL'     : 'CALI', # Hydraulic caliper (Schlumberger)
    'CAL'      : 'CALI', # Generic caliper
    'DCAV'     : 'CALI', # Density caliper average (LWD — 3-BRSA-1367)
}

STANDARD_COLUMNS = ['DEPTH', 'GR', 'RT90', 'RHOB', 'NPHI', 'DT', 'CALI', 'Well_Name']


In [ ]:
wells = {}

for las_file in sorted(LAS_FOLDER.glob('*.las')):
    well_name = las_file.stem
    df = lasio.read(str(las_file)).df().reset_index()
    df['Well_Name'] = well_name
    df = df.rename(columns=COLUMN_RENAME)
    df = df[[col for col in STANDARD_COLUMNS if col in df.columns]]
    wells[well_name] = df

print(f'Total files loaded: {len(wells)}')
print('Wells:')
for name in wells.keys():
    print(f'  {name}')

## Utility Functions

Helper functions used across multiple processing steps in this notebook.

In [ ]:
def detect_phase_boundary(df, min_jump_in=2.0):
    """
    Detects the boundary between drilling phases from a step change
    in the CALI curve. Two methods are applied in sequence:

    1. Abrupt jump: largest point-to-point variation in the smoothed CALI
       (window=5). Captures sharp transitions between phases with different
       bit sizes.

    2. Permanent midpoint crossover: detects gradual transitions in wells
       with a bimodal CALI distribution (Q10–Q90 gap >= 3 inches).
       Works in both directions — smaller to larger bit and vice versa.

    Returns the boundary depth or None if not detected.
    min_jump_in: minimum jump threshold for Method 1 (default 2 in).
    """
    df_sorted = df.sort_values('DEPTH').reset_index(drop=True)
    cali = df_sorted['CALI']

    # Method 1: abrupt jump
    cali_smooth = cali.rolling(5, center=True, min_periods=3).mean()
    diff = cali_smooth.diff().abs()
    idx_max = diff.idxmax()
    if diff[idx_max] >= min_jump_in:
        return df_sorted.loc[idx_max, 'DEPTH']

    # Method 2: bimodal distribution
    q10 = cali.quantile(0.10)
    q90 = cali.quantile(0.90)
    if (q90 - q10) < 3.0:
        return None

    midpoint = (q10 + q90) / 2
    cali_smooth50 = cali.rolling(50, center=True, min_periods=10).mean()

    # Detect transition in both directions
    # Case A: starts below midpoint and crosses above (smaller → larger bit)
    above = cali_smooth50 > midpoint
    if not above.iloc[0]:
        for i in range(len(above) - 50):
            if above.iloc[i] and above.iloc[i:i+50].all():
                return df_sorted.loc[i, 'DEPTH']

    # Case B: starts above midpoint and crosses below (larger → smaller bit)
    below = cali_smooth50 < midpoint
    if not below.iloc[0]:
        for i in range(len(below) - 50):
            if below.iloc[i] and below.iloc[i:i+50].all():
                return df_sorted.loc[i, 'DEPTH']

    return None


def iqr_filter_cali(df):
    """Applies IQR filter on CALI. Returns (filtered_df, n_removed)."""
    Q1 = df['CALI'].quantile(0.25)
    Q3 = df['CALI'].quantile(0.75)
    iqr_val = Q3 - Q1
    lower = Q1 - 1.5 * iqr_val
    upper = Q3 + 1.5 * iqr_val
    filtered = df[(df['CALI'] >= lower) & (df['CALI'] <= upper)]
    return filtered, len(df) - len(filtered)


# Raw Data Visualization

Plotting raw well curves to identify potential issues such as gaps, scale inconsistencies, and borehole problems.

In [ ]:
for well_name, df in wells.items():
    print(f'Well: {well_name}')
    plot_well(df).show()

**Observations:**
- Well `1-BRSA-1013-SES` exhibits gaps in the sonic curve (DT) and an anomalous peak
  in the neutron porosity curve (NPHI). The DT gaps will be corrected by linear interpolation;
  the NPHI peak will be removed by the physical range filter.
- In some wells, the NPHI was exported as a fraction (0–1) rather than percentage units (0–100 pu)
- All wells were drilled in multiple phases; however, for wells `3-BRSA-1180D-SES`,
  `3-BRSA-1194-SES`, `9-BRSA-1278A-SES` and `9-BRSA-1280D-SES`, the available log data
  spans two phases with different bit sizes, producing a visible step in the CALI curve
  at the phase boundary — this will be addressed by applying the IQR filter independently
  to each phase.
- Well `3-BRSA-1194-SES` exhibits a phase offset in the NPHI curve, and well `9-BRSA-1278A-SES`
  exhibits a systematic shift in the GR curve between phases. Both will be corrected by
  inter-phase normalisation prior to IQR filtering. If the correction is unsatisfactory,
  the affected section will be removed.

## Raw Data Quality Control — Curve Range Check

Before any processing, a quick sanity check is performed on the min/max values
of all standard curves across every well.

This step helps identify common data issues such as:
- **Unmasked null values** (e.g. -999.25 or -9999.25 not recognized by lasio)
- **Scale inconsistencies** (e.g. NPHI in fraction instead of percentage)
- **Instrument spikes** (isolated extreme readings)
- **Missing curves** (shown as `---`)

Expected ranges for reference:

| Curve | Unit | Typical min | Typical max |
|-------|------|-------------|-------------|
| GR    | gAPI | 0           | 300         |
| RT90  | Ω·m  | 0.1         | 10,000      |
| RHOB  | g/cm³| 1.8         | 3.0         |
| NPHI  | %    | -5          | 100         |
| DT    | µs/ft| 40          | 200         |
| CALI  | in   | 4           | 30          |

Any value outside these ranges warrants investigation before proceeding.

In [ ]:
qc_cols = ['GR', 'RT90', 'RHOB', 'NPHI', 'DT', 'CALI']

col_w  = 18
well_w = 25

line1 = f"{'Well':<{well_w}s}"
line2 = ' ' * well_w
sep   = '-' * well_w

for col in qc_cols:
    line1 += f"  {col:^{col_w}s}"
    line2 += f"  {'min':>{col_w//2 - 1}s} {'max':<{col_w//2}s}"
    sep   += '  ' + '-' * col_w

print(line1)
print(line2)
print(sep)

for well_name, df in sorted(wells.items()):
    row = f"{well_name:<{well_w}s}"
    for col in qc_cols:
        if col in df.columns:
            mn = df[col].min()
            mx = df[col].max()
            row += f"  {mn:{col_w//2 - 1}.2f} {mx:<{col_w//2}.2f}"
        else:
            row += f"  {'---':>{col_w//2 - 1}s} {'---':<{col_w//2}s}"
    print(row)

## NPHI scale correction

Some wells have NPHI exported in fraction (0–1) instead of percentage (0–100 pu).
The correction uses the **median** as criterion — isolated spikes above 1.0 would
cause a `max ≤ 1` check to fail even when the curve is genuinely in the 0–1 scale.
Wells with median NPHI ≤ 1.0 are multiplied by 100.

In [ ]:
for well_name, df in wells.items():
    if 'NPHI' not in df.columns:
        continue
    median_nphi = df['NPHI'].median()
    if median_nphi <= 1.0:
        wells[well_name]['NPHI'] = df['NPHI'] * 100
        print(f'{well_name}: NPHI × 100 '
              f'(median={median_nphi:.3f} → 0–1 scale detected)')

print('\nNPHI scale correction complete.')
print()
print('Final check (should be between ~0 and ~100):')
for well_name, df in sorted(wells.items()):
    if 'NPHI' in df.columns:
        mn, mx = df['NPHI'].min(), df['NPHI'].max()
        flag = ' <- CHECK' if mx < 2 or mx > 150 else ''
        print(f'  {well_name:<35s} NPHI: [{mn:7.2f}, {mx:7.2f}]{flag}')


# Data Quality Control & Preprocessing

This section covers all operations applied to the raw well data before modelling:
physical range filtering, well-specific GR normalisation, DT gap interpolation,
caliper-based IQR filtering, and final null removal.
Each step targets a distinct data quality issue; all null-generating operations
converge in a single `dropna` at the end of the pipeline.

## Physical Range Filter

Values outside physically plausible limits are set to `NaN` before any other
processing. These readings are instrument artefacts — spikes, bit errors or
tool dropouts — that cannot represent real formation properties.

Affected samples are not deleted here; they become `NaN` and are removed
later in the unified `dropna` step together with all other null values.

| Curve | Unit   | Min   | Max    | Notes                                          |
|-------|--------|------:|-------:|------------------------------------------------|
| GR    | gAPI   |     0 |    300 | Values >300 are tool/K-spike artefacts         |
| RT90  | Ω·m    |  0.01 | 30 000 | Negative and zero resistivity are non-physical |
| RHOB  | g/cm³  |   1.0 |    3.5 | Below air density or above rock maximum        |
| NPHI  | pu     |    -5 |    100 | Tool physically capped at ~100 pu              |
| DT    | µs/ft  |    40 |    250 | Faster than steel / slower than water          |
| CALI  | in     |     4 |     30 | Bit size to maximum expected washout           |

In [ ]:
PHYSICAL_LIMITS = {
    'GR'  : (0,    300),
    'RT90': (0.01, 30000),
    'RHOB': (1.0,  3.5),
    'NPHI': (-5,   100),
    'DT'  : (40,   250),
    'CALI': (4,    30),
}

total_clipped = 0
for well_name, df in wells.items():
    well_clipped = 0
    for col, (lo, hi) in PHYSICAL_LIMITS.items():
        if col not in df.columns:
            continue
        mask = (df[col] < lo) | (df[col] > hi)
        n = mask.sum()
        if n > 0:
            wells[well_name].loc[mask, col] = np.nan
            well_clipped += n
            print(f'  {well_name} | {col}: {n} values outside [{lo}, {hi}] → NaN')
    total_clipped += well_clipped

print(f'\nTotal values clipped: {total_clipped:,}')

## Interpolating DT gaps

Isolated null values in the sonic curve (DT) are filled using linear interpolation
between neighboring values, with a maximum limit of 10 consecutive samples.

Gaps longer than 10 samples are intentionally left as NaN and will be removed
in the null-removal step below. This avoids generating artificial ramps in sections
with no real measurement — in particular at the phase boundary of two-phase wells
(`9-BRSA-1278A-SES` and `9-BRSA-1280D-SES`), where the transition between
two wireline runs naturally produces a gap with no DT coverage.

In [ ]:
MAX_GAP_SAMPLES = 10  # gaps larger than this are not interpolated

for well_name, df in wells.items():
    n_nulls = df['DT'].isnull().sum()
    if n_nulls == 0:
        continue

    # Interpolate only short gaps (≤ MAX_GAP_SAMPLES consecutive samples)
    dt_interp = df['DT'].interpolate(method='linear', limit=MAX_GAP_SAMPLES)
    n_filled = n_nulls - dt_interp.isnull().sum()
    n_kept   = dt_interp.isnull().sum()
    wells[well_name]['DT'] = dt_interp

    print(f'{well_name}: {n_nulls} nulls → {n_filled} interpolated, '
          f'{n_kept} kept as NaN (long gap → removed by dropna)')


## Merging 3-BRSA-1297-SES + 3-BRSA-1297A-SES

Justification: same physical well — identical coordinates (~10 m), same KB (25 m), same GL (−2 135 m), same operator (Petrobras/Schlumberger).
Distinct APIs required by ANP regulatory (different rigs in different phases).
Continuous curves with no discontinuity at the junction (~4 031–4 100 m);
DT at the transition: 102.2 vs 103.0 µs/ft. Gap of 69 m with no overlap.
Result: 27 wells in the dataset (was 28).

In [ ]:
WELL_1297  = '3-BRSA-1297-SES'
WELL_1297A = '3-BRSA-1297A-SES'

if WELL_1297 in wells and WELL_1297A in wells:
    df_combined = (
        pd.concat([
            wells[WELL_1297],
            wells[WELL_1297A].assign(Well_Name=WELL_1297)
        ], ignore_index=True)
        .sort_values('DEPTH')
        .reset_index(drop=True)
    )
    wells[WELL_1297] = df_combined
    del wells[WELL_1297A]

    print(f'Merge complete: {WELL_1297}')
    print(f'Interval  : {df_combined["DEPTH"].min():.1f} – '
          f'{df_combined["DEPTH"].max():.1f} m')
    print(f'N samples : {len(df_combined):,}')
    print(f'Wells in dataset: {len(wells)} (was 28)')
else:
    print('One of the wells not found — check names in the wells dictionary.')


## Saving Pre-IQR Dataset

The dataset is saved at this point — after physical range filtering,
NPHI scale correction and DT interpolation, but **before** the caliper
IQR filter. This file (`wells_no_iqr.csv`) is used by Notebook 03
to evaluate the impact of the IQR filter on model performance
(Modelo A — sem IQR vs Modelo B — com IQR).

In [ ]:
wells_no_iqr = pd.concat(wells.values(), ignore_index=True)
print(f'Total samples (pre-IQR): {len(wells_no_iqr):,}')
print(f'Wells: {wells_no_iqr["Well_Name"].nunique()}')
wells_no_iqr.to_csv('../data/processed/wells_no_iqr.csv', index=False)
print('Saved: ../data/processed/wells_no_iqr.csv')

## Caliper Quality Control (IQR Filter)

The caliper curve (CALI) measures borehole wall integrity. Collapsed or washed-out
sections produce unreliable density (RHOB) and porosity (NPHI) measurements.

The [IQR Statistical Method](https://en.wikipedia.org/wiki/Interquartile_range)
is used to identify and remove depth intervals with anomalous caliper values
(> 1.5 × IQR from quartiles).

**Per-phase detection:** Some wells were drilled in two phases with different bit sizes,
producing two distinct CALI distributions in the same well. Applying IQR to the full well
would eliminate one phase entirely. The filter therefore detects phase boundaries
**automatically** in all wells via a step change in the smoothed CALI curve
(minimum jump of 2 inches). If a boundary is found, IQR is applied independently
to each phase; otherwise the well is filtered as a whole.


In [ ]:
wells_filtered = {}

for well_name, df in wells.items():
    boundary = detect_phase_boundary(df)

    if boundary is not None:
        # Two-phase well — IQR per phase
        phase_a = df[df['DEPTH'] <  boundary]
        phase_b = df[df['DEPTH'] >= boundary]
        filt_a, n_rem_a = iqr_filter_cali(phase_a)
        filt_b, n_rem_b = iqr_filter_cali(phase_b)
        df_filtered = pd.concat([filt_a, filt_b], ignore_index=True)
        n_removed = n_rem_a + n_rem_b
        print(f'{well_name}: 2 phases | boundary ≈ {boundary:.0f}m '
              f'| phase A {len(phase_a):,} samples (−{n_rem_a}) '
              f'| phase B {len(phase_b):,} samples (−{n_rem_b})')
    else:
        # Single-phase well — IQR on the whole well
        df_filtered, n_removed = iqr_filter_cali(df)
        if n_removed > 0:
            print(f'{well_name}: {n_removed:,} samples removed by CALI IQR filter')

    wells_filtered[well_name] = df_filtered

print(f'\nTotal wells: {len(wells_filtered)}')


In [ ]:
print(f"{'Well':<35s} {'N samples':>10s}  {'NPHI min':>8s}  {'NPHI max':>8s}  {'CALI min':>8s}  {'CALI max':>8s}  {'GR min':>8s}  {'GR max':>8s}")
print('-' * 110)
for well_name, df in sorted(wells_filtered.items()):
    print(f"{well_name:<35s} {len(df):>10,}  "
          f"{df['NPHI'].min():>8.2f}  {df['NPHI'].max():>8.2f}  "
          f"{df['CALI'].min():>8.2f}  {df['CALI'].max():>8.2f}  "
          f"{df['GR'].min():>8.2f}  {df['GR'].max():>8.2f}")


## GR Phase Normalization — 9-BRSA-1278A-SES

The well `9-BRSA-1278A-SES` was drilled in two phases. A systematic offset
in the ECGR curve is visible at the phase boundary (~4 450 m), where the
level of the deeper phase (12") is approximately double that of the shallower
phase (18"). The shallower phase GR distribution is consistent with the
regional pattern observed across the other wells in the dataset
(median ~100 gAPI), indicating that the 18" phase is correctly calibrated.

**Normalization method:** the 12" phase is shifted so that its P50 matches
the P50 of the 18" phase (additive offset). A P10/P90 linear rescaling is
also computed for comparison. Both offsets are printed below so that the
most appropriate correction can be selected.

**Justification:** regional GR normalization using a reference well or a
reference phase is standard practice in petrophysical studies (Shier, 2004).
The 18" phase serves as the regional anchor because its distribution matches
the basin-wide GR pattern. The correction is additive (no rescaling of
variance), preserving the relative contrasts within the 12" phase.

In [ ]:
WELL_1278A           = '9-BRSA-1278A-SES'
PHASE_BOUNDARY_1278A = 4480
WINDOW_M             = 100   # meters above and below the boundary

df_1278a = wells_filtered[WELL_1278A].sort_values('DEPTH').copy()

# Window immediately above the boundary (18" phase)
window_18 = df_1278a[
    (df_1278a['DEPTH'] >= PHASE_BOUNDARY_1278A - WINDOW_M) &
    (df_1278a['DEPTH'] <  PHASE_BOUNDARY_1278A)
]['GR'].dropna()

# Window immediately below the boundary (12" phase), excluding spikes
# Skip the first 50 m of unstable transition
window_12 = df_1278a[
    (df_1278a['DEPTH'] >= PHASE_BOUNDARY_1278A + 150) &
    (df_1278a['DEPTH'] <  PHASE_BOUNDARY_1278A + WINDOW_M + 150)
]['GR'].dropna()

p50_18 = window_18.quantile(0.50)
p50_12 = window_12.quantile(0.50)
offset_p50 = p50_18 - p50_12

print(f'18" phase window [{PHASE_BOUNDARY_1278A-WINDOW_M:.0f}–{PHASE_BOUNDARY_1278A:.0f}m]: P50 = {p50_18:.1f} gAPI  (n={len(window_18)})')
print(f'12" phase window [{PHASE_BOUNDARY_1278A+150:.0f}–{PHASE_BOUNDARY_1278A+150+WINDOW_M:.0f}m]: P50 = {p50_12:.1f} gAPI  (n={len(window_12)})')
print(f'Calculated offset: {offset_p50:+.2f} gAPI')

# Apply offset to the entire 12" phase
mask_12 = wells[WELL_1278A]['DEPTH'] >= PHASE_BOUNDARY_1278A
wells_filtered[WELL_1278A].loc[mask_12, 'GR'] = (
    wells_filtered[WELL_1278A].loc[mask_12, 'GR'] + offset_p50
)

print(f'\nPost-correction check:')
print(f'  18" phase P50: {p50_18:.1f} gAPI')
p50_12_corr = wells_filtered[WELL_1278A][mask_12]['GR'].dropna().quantile(0.50)
print(f'  12" phase P50 (full): {p50_12_corr:.1f} gAPI')


In [ ]:
# Normalized GR visualization — 9-BRSA-1278A-SES
print('9-BRSA-1278A-SES — GR after phase normalization')
plot_well(wells_filtered[WELL_1278A]).show()


In [ ]:
mask_transition = (
    (wells_filtered[WELL_1278A]['DEPTH'] >= PHASE_BOUNDARY_1278A) &
    (wells_filtered[WELL_1278A]['DEPTH'] <  4630)
)
wells_filtered[WELL_1278A].loc[mask_transition, 'GR'] = np.nan

n = mask_transition.sum()
print(f'GR → NaN for {n} samples in the transition interval '
      f'[{PHASE_BOUNDARY_1278A}–4630m]')

## NPHI Phase Normalisation — 3-BRSA-1194-SES

The composite log for well `3-BRSA-1194-SES` indicates that the NPHI curve
is unreliable above 4 700 m — the official record only considers the
interval below that depth as valid. Above 4 550 m the curve is absent;
between 4 550 m and 4 700 m it is present but affected by an instrumental
offset that makes it untrustworthy.

The apparent difference in NPHI between the two depth intervals is not an
artefact: the upper section is dominated by shale and carbonate
(high porosity index), while the lower section contains frequent sandstone
intercalations with oil shows and formation tests (lower porosity index).
This is a genuine lithological contrast, not a calibration error.

**Decision:** NPHI values above 4 700 m are set to `NaN` and will be
removed by the final `dropna` step, retaining only the reliable interval.

In [ ]:
WELL_1194        = '3-BRSA-1194-SES'
NPHI_VALID_FROM  = 4700   # meters — lower bound of the reliable zone

mask_invalid = wells_filtered[WELL_1194]['DEPTH'] < NPHI_VALID_FROM
n = mask_invalid.sum()
wells_filtered[WELL_1194].loc[mask_invalid, 'NPHI'] = np.nan

print(f'NPHI → NaN for {n} samples above {NPHI_VALID_FROM}m ({WELL_1194})')
print('These samples will be removed by the final dropna.')
print()
print('Post-correction check:')
valid = wells_filtered[WELL_1194][~mask_invalid]['NPHI'].dropna()
print(f'  n valid   = {len(valid):,}')
print(f'  NPHI min  = {valid.min():.1f} pu')
print(f'  NPHI P50  = {valid.median():.1f} pu')
print(f'  NPHI max  = {valid.max():.1f} pu')


In [ ]:
# Corrected NPHI visualization — 3-BRSA-1194-SES
print('3-BRSA-1194-SES — NPHI after removal of the invalid interval')
plot_well(wells_filtered[WELL_1194]).show()


## Removing Remaining Null Values

After all previous steps, any rows that still contain `NaN` in any column are removed.
This is the single `dropna` pass that aggregates the effect of:
- Physical range filter (values outside plausible limits → NaN)
- GR normalisation (no NaN generated, but GR values shifted)
- DT interpolation (long gaps intentionally kept as NaN)
- IQR caliper filter (already removed bad rows; residual NaN from other curves)


In [ ]:
for well_name in wells_filtered:
    n_before = len(wells_filtered[well_name])
    wells_filtered[well_name] = wells_filtered[well_name].dropna()
    n_removed = n_before - len(wells_filtered[well_name])
    if n_removed > 0:
        print(f'{well_name}: {n_removed} rows with NaN removed')

print('Null removal complete.')

In [ ]:
for well_name, df in wells_filtered.items():
    print(f'Well: {well_name}')
    plot_well(df).show()

# Final Dataset

Concatenating all preprocessed and filtered wells into a single DataFrame
and saving as `wells_iqr.csv`.

In [ ]:
wells_iqr = pd.concat(wells_filtered.values(), ignore_index=True)

print(f'Total samples (post-filter): {len(wells_iqr):,}')
print(f'Nulls: {wells_iqr.isnull().sum().sum()}')
print(f'Duplicates: {wells_iqr.duplicated().sum()}')
print()
wells_iqr.describe()

In [ ]:
wells_iqr.to_csv('../data/processed/wells_iqr.csv', index=False)
print('Saved: ../data/processed/wells_iqr.csv')